<a href="https://colab.research.google.com/github/OrastaRakhmatullayeva/ML-projects/blob/main/Pipeline_and_cross_validation_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.Data leakage nima? Bitta oddiy misol keltiring.

2.Nega StandardScaler'ni butun ma'lumotga emas, faqat train qismiga fit qilish kerak?

3.Pipeline'ning 3 ta asosiy foydasini yozing.

4.K-fold cross-validation qanday ishlaydi? cv=5 bo'lsa, model necha marta o'qitiladi?

5.Nega bitta train/test bo'linishi o'rniga cross-validation ishonchliroq hisoblanadi?


ANSWERS:

1.Data leakage bu ko'rilishi va o'rganilishi kk bo'lmagan malumotlarning test qismiga aralashib ketishi va overfitting ga olib keladi.

misol, talaba imtihonga tushushi mukin bo'lgaan malumotni va tushushi aniq bo'lgan savollarnii o'rganadi. natijada testni alo darajada bajaradi lekin fan bo'yicha tushunchasi bo'lmaydi

2.Chunki Scaling qilayotganimizda std va mean hisoblanadi, agar butun data ni scale qilsak, u yerda testdagi malumotlar ham arlashib ketadi va data leakage ga olib keladi.

3.pipeline ixcham, tartibli va data leakage ni oldini oladi.

4.k-fold cross-vaalidation bu datasetni k ta qismga bo'lib o'rganuvchi paramaterdir. cv=5 bo'lsa datani 5 ga bo'ladi, k-1 training uchun qolgani esa test uchun, 5 marta o'rganib, averaging/voting orqali final answer olinadi.

5.Cross-validationda har bir ma'lumot nuqtasi ham train, ham test bosqichida qatnashad. train test ga bo'lsak, bazi dataning qismlari ishtirok etmasligi, bazida o'ta zo'r bazida esa yomon natija chiqarishi mumkin.


# task 2 finding error:


In [15]:
# Code with mistake

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)

# Scaler butun ma'lumotga fit qilinmoqda
X_scaled = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("Test accuracy:", model.score(X_test, y_test))


Test accuracy: 0.9736842105263158


In [16]:
# Code with mistake

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)

# Scaler FAQAT TRAIN qismiga fit qilinmoqda
sc=StandardScaler().fit(X_train)


X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(sc.transform(X_train), y_train)
print("Test accuracy:", model.score(X_test, y_test))


Test accuracy: 0.9824561403508771


# task 3

In [17]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

# TODO: X, y ni train/test ga bo'ling
# TODO: pipe.fit(...) va pipe.score(...) bilan natijani chiqaring


Pipeline ichidagi bosqichlar nomlarini (pipe.named_steps) chop eting.

Pipeline'ga SimpleImputer(strategy="mean") bosqichini scaler'dan oldin qo'shing. Nima o'zgardi?

Pipeline'ni pipe.predict(X_test[:5]) orqali ishlating va real y_test[:5] bilan solishtiring.


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

pipe=Pipeline(
    steps=[
        ("sc",StandardScaler()),

        ("impute",SimpleImputer(strategy="mean")),
        ("clf",LogisticRegression(max_iter=1000))
    ]
)

pipe.fit(X_train,y_train)
print(pipe.score(X_test,y_test))

print(pipe.predict(X_test[:5]))
print((y_test[:5]))

0.9824561403508771
[0 1 0 1 0]
[0 1 0 1 0]


# task 4


In [19]:
from sklearn.model_selection import cross_val_score
import numpy as np

scores = cross_val_score(pipe, X, y, cv=5)
print("Har bir fold:", scores)
print("O'rtacha:", scores.mean())
print("Std:", scores.std())


Har bir fold: [0.98245614 0.98245614 0.97368421 0.97368421 0.99115044]
O'rtacha: 0.9806862288464524
Std: 0.006539441283506109



a) cv=3, cv=5 va cv=10 uchun o'rtacha aniqlik va standart og'ishni hisoblang, jadvalga yozing.

b) Nima uchun cross_val_score'ga scaler'siz emas, aynan pipe berilyapti? (2–3 gap)

c) Bitta train/test natijasi (3-vazifa) va CV o'rtacha natijasini solishtiring. Qaysi biriga ko'proq ishonasiz va nega?


In [20]:
# a)


scores = cross_val_score(pipe, X, y, cv=3)
print("Har bir fold:", scores)
print("O'rtacha:", scores.mean())
print("Std:", scores.std())


Har bir fold: [0.97894737 0.97368421 0.97354497]
O'rtacha: 0.975392184164114
Std: 0.0025145374736826936


In [21]:

scores = cross_val_score(pipe, X, y, cv=10)
print("Har bir fold:", scores)
print("O'rtacha:", scores.mean())
print("Std:", scores.std())


Har bir fold: [0.98245614 0.98245614 0.98245614 0.96491228 0.98245614 0.98245614
 0.94736842 1.         1.         0.98214286]
O'rtacha: 0.9806704260651629
Std: 0.01456955548732776


b)Ma'lumotlarni scalerni alohida qo'llab, keyin kross-validatsiyaga berilsa, Data Leakage bo'ladi. Pipeline ishlatilganda, har bir fold uchun scaler faqat o'sha foldning train qismida fit() qilinadi va val/test qismiga transform() qilinadi.


c) albatta cv natijasiga.

O'rtacha cv natijasiga qarasak, ~97% atrofida , ammo hamma malumotni o'rgangan 3,5,10 bo'lib. Lekin train test splitda 98% bo'lsa ham , bunda 100 foiz ishhona olamiz deb ayta olmaymiz, balki train ga ham test gaa ham eng yaxshi va toza malumotlar tushub qolgan bo'lsa, u almost perfect natija chiqarayotgan bo'lsihi mumkin

# task 5

In [22]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "RandomForest": RandomForestClassifier(random_state=42),
}

for name, clf in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])
    scores = cross_val_score(pipe, X, y, cv=5)
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")


LogReg: 0.981 (+/- 0.007)
KNN: 0.965 (+/- 0.010)
RandomForest: 0.956 (+/- 0.023)


In [23]:
import pandas as pd

results = []
for name, clf in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])
    scores = cross_val_score(pipe, X, y, cv=5)
    results.append({
        "Model": name,
        "O'rtacha": round(scores.mean(), 3),
        "Std": round(scores.std(), 3),
    })

df = pd.DataFrame(results).sort_values("O'rtacha", ascending=False)
print(df.to_string(index=False))

       Model  O'rtacha   Std
      LogReg     0.981 0.007
         KNN     0.965 0.010
RandomForest     0.956 0.023


Qaysi model eng yaxshi? Std kichik bo'lishi nimani anglatadi?

hozircha Logistic Regression modeli yaxshi natija qayd qilmoqda


RandomForest uchun scaler kerakmi? O'z fikringizni izohlang (bu savolga to'g'ri javob topish uchun qidiring).


YO'q. random forest savol berish orqali treelarni quradi, unga categoric malumotlar bo'lsa ham ahamiytasiz

# task 6

In [24]:
df = pd.DataFrame({
  "yosh":   [25,32,None,45,52,23,36,41,29,60,38,27],
  "maosh":  [500,800,650,None,1200,450,900,1000,700,1500,850,600],
  "shahar": ["Toshkent","Samarqand","Toshkent","Buxoro","Samarqand","Buxoro",
             "Toshkent","Samarqand","Buxoro","Toshkent","Samarqand","Buxoro"],
  "kredit_oldi": [0,1,0,1,1,0,1,1,0,1,1,0]
})
X = df.drop("kredit_oldi", axis=1)
y = df["kredit_oldi"]



Vazifa: ColumnTransformer va Pipeline yordamida to'liq ishlov berish zanjirini quring:

Raqamli ustunlar (yosh, maosh): SimpleImputer(strategy="median") → StandardScaler

Kategorial ustun (shahar): OneHotEncoder(handle_unknown="ignore")

Oxirida LogisticRegression; natijani cross_val_score(..., cv=3) bilan baholang.


In [33]:
num=X.select_dtypes(include='number').columns.tolist()
cat=X.select_dtypes(exclude='number').columns.tolist()

In [34]:
num_pipe=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scalar',StandardScaler())
    ]
)

In [35]:
from sklearn.preprocessing import OneHotEncoder
cat_pipe=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('One hot',OneHotEncoder(handle_unknown='ignore'))
    ]
)

In [36]:
from sklearn.compose import ColumnTransformer

In [37]:
pre=ColumnTransformer(
    transformers=[
        ("num",num_pipe,num),
        ('cat',cat_pipe,cat)
    ]
)

pre

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scalar', StandardScaler())]),
                                 ['yosh', 'maosh']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('One hot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['shahar'])])

In [56]:
clf=Pipeline(
    steps=[
        ('pre',pre),
        ("model",LogisticRegression(max_iter=1000))
    ]
)

In [57]:

scores = cross_val_score(clf, X, y, cv=3)
print("Foldlar bo'yicha:", scores.round(3))
print(f"O'rtacha: {scores.mean():.3f} std: {scores.std():.3f}")

Foldlar bo'yicha: [0.75 1.   1.  ]
O'rtacha: 0.917 std: 0.118
